In [1]:
# --------------------------
# author: Brian Kyanjo
# date: 2026-02-09
# description: This is an ICESEE-GHUB GUI notebook for rununing
#              ICESEE on cloud and cluster environments. It is 
#              designed to be run in a Jupyter Book environment.
# --------------------------

from IPython.display import HTML
HTML("""
<style>
/* Hide code inputs in Jupyter Book pages */
div.cell_input {display:none;}
</style>
""")

## ICESEE GUI

In [ ]:
from __future__ import annotations

import os, sys, subprocess, shutil
from pathlib import Path
from datetime import datetime
import yaml

import ipywidgets as W
from IPython.display import display, Image

# ============================================================
# 0) Paths
# ============================================================
def find_repo_root():
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "external" / "ICESEE").exists() and (p / "icesee_jupyter_book").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not locate repo root containing external/ICESEE and icesee_jupyter_book.")

REPO = find_repo_root()
BOOK = REPO / "icesee_jupyter_book"
EXT  = REPO / "external" / "ICESEE"

# ============================================================
# 1) Example registry (register examples here to show in dropdown)
# ============================================================
EXAMPLES = {
    "Lorenz-96 (fully runnable in GHUB)": dict(
        enabled=True,
        base=EXT / "applications" / "lorenz_model" / "examples" / "lorenz96",
        run_script="run_da_lorenz96.py",
        params="params.yaml",
        report_nb="read_results.ipynb",
        assets=["_modelrun_datasets"],   # copy into run dir for report, if needed
        model_name="lorenz",             # used for expected h5 name
        figures_dir="figures",           # where read_results typically saves images
    ),
    "ISSM (under development)":     dict(
        enabled=False,
        base=EXT / "applications" / "issm_model" / "examples" / "ISMIP_Choi",
        run_script="run_da_issm.py",
        params="params.yaml",
        report_nb="read_results.m",
        assets=["_modelrun_datasets"],
        model_name="issm",
        figures_dir="figures",
    ),
    "Flowline (under development)": dict(
        enabled=False, 
        base=EXT / "applications" / "flowline_model" / "examples" / "flowline_1d",
        run_script="run_da_flowline.py",
        params="params.yaml",
        report_nb="read_results.ipynb",
        assets=["_modelrun_datasets"],
        model_name="flowline",
        figures_dir="figures",
    ),
    "Icepack (under development)":  dict(
        enabled=False, 
        base=EXT / "applications" / "icepack_model" / "examples" / "synthetic_ice_stream",
        run_script="run_da_icepack.py",
        params="params.yaml",
        report_nb="read_results.ipynb",
        assets=["_modelrun_datasets"],
        model_name="icepack",
        figures_dir="figures",
    ),
    
}

def enabled_names():
    return [k for k,v in EXAMPLES.items() if v.get("enabled", False)]

# ============================================================
# 2) YAML helpers
# ============================================================
def load_yaml(p: Path) -> dict:
    return yaml.safe_load(p.read_text(encoding="utf-8")) or {}

def dump_yaml(d: dict, p: Path) -> None:
    p.write_text(yaml.safe_dump(d, sort_keys=False), encoding="utf-8")

def deep_update(d, u):
    for k, v in (u or {}).items():
        if isinstance(v, dict) and isinstance(d.get(k), dict):
            deep_update(d[k], v)
        else:
            d[k] = v
    return d

# ============================================================
# 3) Find runner + template
# ============================================================
def find_run_script(cfg: dict) -> Path:
    base = cfg["base"]
    rs = cfg.get("run_script")
    if rs and (base / rs).exists():
        return base / rs
    cands = list(base.rglob("run_da_*.py")) + list(base.rglob("run_*.py"))
    cands = [c for c in cands if c.is_file()]
    if not cands:
        raise FileNotFoundError(f"No run script found under {base}")
    cands.sort(key=lambda x: len(str(x)))
    return cands[0]

def find_params_template(cfg: dict) -> Path:
    wrapper_template = BOOK / "params.yaml"
    if wrapper_template.exists():
        return wrapper_template
    base = cfg["base"]
    p = base / (cfg.get("params") or "params.yaml")
    if p.exists():
        return p
    cands = list(base.rglob("params.yaml"))
    if not cands:
        raise FileNotFoundError(f"No params.yaml found under {base}")
    cands.sort(key=lambda x: len(str(x)))
    return cands[0]

def find_report_notebook(cfg: dict) -> Path | None:
    nb = cfg.get("report_nb")
    if not nb:
        return None
    p = cfg["base"] / nb
    return p if p.exists() else None

# ============================================================
# 4) Widget factory from YAML
# ============================================================
def widget_for(key: str, val):
    if isinstance(val, str):
        if key.lower() == "filter_type":
            opts = ["EnKF", "DEnKF", "EnTKF", "EnRSKF"]
            return W.Dropdown(options=opts, value=val if val in opts else opts[0], layout=W.Layout(width="100%"))
        if key.lower() in {"parallel_flag", "parallel"}:
            opts = ["serial", "MPI", "MPI_model"]
            return W.Dropdown(options=opts, value=val if val in opts else opts[0], layout=W.Layout(width="100%"))
        return W.Text(value=val, layout=W.Layout(width="100%"))

    if isinstance(val, bool):
        return W.Checkbox(value=val)

    if isinstance(val, int) and not isinstance(val, bool):
        return W.IntText(value=val, layout=W.Layout(width="100%"))
    if isinstance(val, float):
        return W.FloatText(value=val, layout=W.Layout(width="100%"))

    if isinstance(val, (list, dict)):
        return W.Textarea(
            value=yaml.safe_dump(val, sort_keys=False).strip(),
            layout=W.Layout(width="100%", height="110px"),
        )

    return W.Text(value=str(val), layout=W.Layout(width="100%"))

def read_widget(w):
    if isinstance(w, W.Textarea):
        try:
            return yaml.safe_load(w.value)
        except Exception:
            return w.value
    if hasattr(w, "value"):
        return w.value
    return None

# ============================================================
# 5) UI state
# ============================================================
example_dd = W.Dropdown(options=enabled_names(), value=enabled_names()[0], layout=W.Layout(width="320px"))

preset_dd  = W.Dropdown(options=["Default"], value="Default", layout=W.Layout(width="320px"))

# Filter algorithm (goes into params.yaml -> enkf-parameters/filter_type)
filter_alg_dd = W.Dropdown(
    options=[("EnKF", "EnKF"), ("DEnKF", "DEnKF"), ("EnTKF", "EnTKF"), ("EnRSKF", "EnRSKF")],
    value="EnKF",
    layout=W.Layout(width="320px")
)

# Output label controls expected result filename: results/<output>-<model>.h5
output_label_dd = W.Dropdown(
    options=[("true-wrong (demo output)", "true-wrong"), ("EnKF (output name)", "enkf")],
    value="true-wrong",
    layout=W.Layout(width="320px"),
)

ens_sl    = W.IntSlider(min=1, max=200, value=30, layout=W.Layout(width="320px"), continuous_update=False)
seed_in   = W.IntText(value=1, layout=W.Layout(width="320px"))

gen_report = W.Checkbox(value=True, description="Generate report (read_results.ipynb)")
open_latest= W.Checkbox(value=False, description="After run: open latest run folder")

run_btn   = W.Button(description="Run", button_style="success", icon="play")
clear_btn = W.Button(description="Clear", button_style="", icon="trash")

status_chip = W.HTML("<span class='icesee-status icesee-idle'>Idle</span>")

log_out = W.Output(layout=W.Layout(border="1px solid rgba(0,0,0,.12)", padding="10px", height="220px", overflow="auto"))
results_out = W.Output(layout=W.Layout(border="1px solid rgba(0,0,0,.12)", padding="10px", height="260px", overflow="auto"))

# params accordion (auto-built)
params_accordion = None
PARAMS0 = {}
WIDGETS = {}
EXTRA_YAML = {}
RUN_SCRIPT = None
TEMPLATE = None
REPORT_NB = None

# ============================================================
# 6) Build params UI from template
# ============================================================
def build_params_ui(template_path: Path):
    global params_accordion, PARAMS0, WIDGETS, EXTRA_YAML
    PARAMS0 = load_yaml(template_path)
    WIDGETS = {}
    EXTRA_YAML = {}

    children, titles = [], []

    for sec, sec_dict in (PARAMS0 or {}).items():
        titles.append(sec)
        sec_widgets = {}
        rows = []

        if isinstance(sec_dict, dict):
            for k, v in sec_dict.items():
                w = widget_for(k, v)
                sec_widgets[k] = w
                rows.append(W.HBox([
                    W.HTML(f"<div class='icesee-k'>{k}</div>"),
                    w
                ], layout=W.Layout(gap="12px")))

            extra = W.Textarea(
                value="# Add future keys here (YAML)\n",
                layout=W.Layout(width="100%", height="90px")
            )
            EXTRA_YAML[sec] = extra
            rows.append(W.HTML("<div class='icesee-subtle' style='margin-top:6px'>Extra keys (optional)</div>"))
            rows.append(extra)
        else:
            w = W.Textarea(value=yaml.safe_dump(sec_dict, sort_keys=False).strip(),
                           layout=W.Layout(width="100%", height="140px"))
            sec_widgets["__raw__"] = w
            rows.append(w)

        WIDGETS[sec] = sec_widgets
        children.append(W.VBox(rows, layout=W.Layout(gap="8px")))

    params_accordion = W.Accordion(children=children)
    for i, t in enumerate(titles):
        params_accordion.set_title(i, t)

def sync_quick_into_widgets():
    # Try section naming variants
    sec = None
    for candidate in ["enkf-parameters", "enkf_parameters", "enkf"]:
        if candidate in WIDGETS:
            sec = candidate
            break
    if not sec:
        return

    if "Nens" in WIDGETS[sec]:
        WIDGETS[sec]["Nens"].value = int(ens_sl.value)
    if "seed" in WIDGETS[sec]:
        WIDGETS[sec]["seed"].value = int(seed_in.value)
    if "filter_type" in WIDGETS[sec]:
        WIDGETS[sec]["filter_type"].value = str(filter_alg_dd.value)

def build_config_from_widgets() -> dict:
    cfg = {}
    for sec, sw in WIDGETS.items():
        if "__raw__" in sw:
            cfg[sec] = yaml.safe_load(sw["__raw__"].value)
            continue

        cfg[sec] = {}
        for k, w in sw.items():
            if k == "__raw__":
                continue
            cfg[sec][k] = read_widget(w)

        extra = EXTRA_YAML.get(sec)
        if extra:
            txt = extra.value.strip()
            if txt and not txt.startswith("#"):
                extra_obj = yaml.safe_load(txt) or {}
                if isinstance(extra_obj, dict):
                    cfg[sec].update(extra_obj)
                else:
                    cfg[sec]["__extra__"] = extra_obj
    return cfg

# ============================================================
# 7) Run helpers
# ============================================================
def run_dir() -> Path:
    rd = BOOK / "icesee_runs" / datetime.now().strftime("%Y%m%d_%H%M%S")
    rd.mkdir(parents=True, exist_ok=True)
    (rd / "results").mkdir(exist_ok=True)
    (rd / "figures").mkdir(exist_ok=True)
    return rd

def expected_h5_path(rd: Path, cfg: dict) -> Path:
    model_name = cfg.get("model_name", "lorenz")
    return rd / "results" / f"{output_label_dd.value}-{model_name}.h5"

def set_status(state: str):
    cls = {"idle":"icesee-idle","running":"icesee-running","done":"icesee-done","fail":"icesee-fail"}[state]
    label = {"idle":"Idle","running":"Running…","done":"Done","fail":"Failed"}[state]
    status_chip.value = f"<span class='icesee-status {cls}'>{label}</span>"

def force_external_icesee_env():
    external_dir = (REPO / "external").resolve()
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{external_dir}{os.pathsep}{env.get('PYTHONPATH','')}"
    env["PYTHONNOUSERSITE"] = "1"
    return env, external_dir

def mirror_assets_for_report(example_cfg: dict, rd: Path):
    # Some report notebooks expect folders like _modelrun_datasets present.
    base = example_cfg["base"]
    for a in example_cfg.get("assets", []):
        src = base / a
        if src.exists():
            dst = rd / a
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)

def run_report_notebook(example_cfg: dict, rd: Path):
    nb = REPORT_NB
    if not nb or not nb.exists():
        with log_out:
            print("[wrapper] No report notebook configured/found; skipping read_results.")
        return None

    try:
        import papermill as pm
    except Exception as e:
        raise RuntimeError("papermill is required to run read_results.ipynb automatically. "
                           "Install it (pip install papermill) or ask me for an nbclient version.") from e

    nb_out = rd / "report.ipynb"
    mirror_assets_for_report(example_cfg, rd)

    # Execute report with cwd=rd so relative paths resolve:
    # - results/<output>-lorenz.h5
    # - figures/...
    pm.execute_notebook(
        input_path=str(nb),
        output_path=str(nb_out),
        cwd=str(rd),
        log_output=True,
    )
    return nb_out

def refresh_results_preview(rd: Path):
    results_out.clear_output()
    with results_out:
        # prefer real report figures
        fig_dir = rd / "figures"
        pngs = sorted(fig_dir.glob("*.png"))
        if not pngs:
            pngs = sorted((rd / "results").glob("*.png"))

        h5s  = sorted((rd / "results").glob("*.h5"))

        print("Run folder:", rd)
        print(f"Results: {len(h5s)} H5, {len(pngs)} PNG\n")

        if h5s:
            for p in h5s[:10]:
                print(" -", p.name)

        if pngs:
            print("\nFigures:")
            for p in pngs[:6]:
                display(Image(filename=str(p)))
        else:
            print("\nNo figures found yet.")

# ============================================================
# 8) Run
# ============================================================
def run_example():
    example_cfg = EXAMPLES[example_dd.value]

    # push quick knobs into template widgets before writing params.yaml
    sync_quick_into_widgets()

    cfg = build_config_from_widgets()

    rd = run_dir()
    params_path = rd / "params.yaml"
    dump_yaml(cfg, params_path)

    env, external_dir = force_external_icesee_env()

    cmd = [sys.executable, str(RUN_SCRIPT), "-F", str(params_path)]

    set_status("running")
    log_out.clear_output()
    with log_out:
        print("Example :", example_dd.value)
        print("Runner  :", RUN_SCRIPT)
        print("Report  :", REPORT_NB if REPORT_NB else "(none)")
        print("CWD     :", rd)
        print("Command :", " ".join(cmd))
        print("PYTHONPATH(prepended):", external_dir)
        print("-"*70)

    proc = subprocess.Popen(
        cmd,
        cwd=str(rd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    assert proc.stdout is not None
    full_log = []
    for line in proc.stdout:
        full_log.append(line)
        with log_out:
            print(line.rstrip())
    rc = proc.wait()

    log_text = "".join(full_log)
    looks_like_failure = (
        "Traceback (most recent call last)" in log_text
        or "Error in serial run mode" in log_text
        or "UnboundLocalError" in log_text
        or "FileNotFoundError" in log_text
    )

    with log_out:
        print("-"*70)
        print("Return code:", rc)

    # even if rc=0, treat tracebacks as failure
    if rc != 0 or looks_like_failure:
        with log_out:
            if rc == 0 and looks_like_failure:
                print("[GUI] Detected error in output despite rc=0 -> marking as FAILED.")
        set_status("fail")
        refresh_results_preview(rd)
        return

    # Ensure expected H5 exists where report expects it.
    exp_h5 = expected_h5_path(rd, example_cfg)
    if not exp_h5.exists():
        # try to locate it anywhere under external/ICESEE and copy back
        model_name = example_cfg.get("model_name", "lorenz")
        h5_name = f"{output_label_dd.value}-{model_name}.h5"
        candidates = list(EXT.glob(f"**/results/{h5_name}"))
        candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        if candidates:
            exp_h5.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidates[0], exp_h5)
            with log_out:
                print(f"[wrapper] Copied H5 into run folder: {exp_h5.name}")
        else:
            with log_out:
                print(f"[wrapper] WARNING: expected H5 not found: {exp_h5}")
            # still continue; maybe report uses a different filename

    # Run report notebook to generate the REAL figures
    if gen_report.value:
        try:
            with log_out:
                print("[wrapper] Running report notebook (read_results.ipynb)…")
            run_report_notebook(example_cfg, rd)
            with log_out:
                print("[wrapper] Report done.")
        except Exception as e:
            with log_out:
                print("[wrapper] Report failed:", type(e).__name__, e)
            set_status("fail")
            refresh_results_preview(rd)
            return
        
    set_status("done")
    refresh_results_preview(rd)

    with log_out:
        print("\nRun folder:", rd)

# ============================================================
# 9) Rebuild on example change
# ============================================================
def rebuild_for_example(_=None):
    global RUN_SCRIPT, TEMPLATE, REPORT_NB

    cfg = EXAMPLES[example_dd.value]
    RUN_SCRIPT = find_run_script(cfg)
    TEMPLATE   = find_params_template(cfg)
    REPORT_NB  = find_report_notebook(cfg)

    build_params_ui(TEMPLATE)
    params_holder.children = (params_accordion,)

    with log_out:
        print("[Loaded]")
        print("Template:", TEMPLATE)
        print("Runner  :", RUN_SCRIPT)
        print("Report  :", REPORT_NB if REPORT_NB else "(none)")

example_dd.observe(rebuild_for_example, names="value")
run_btn.on_click(lambda b: run_example())
clear_btn.on_click(lambda b: (log_out.clear_output(), results_out.clear_output(), set_status("idle")))

# ============================================================
# 10) Pro UX CSS
# ============================================================
css = """
<style>
.icesee-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial; }
.icesee-title { font-size: 18px; font-weight: 700; margin: 6px 0 4px; }
.icesee-subtitle { color: rgba(0,0,0,.65); margin-bottom: 14px; }
.icesee-card { border: 1px solid rgba(0,0,0,.10); border-radius: 12px; padding: 14px; background: #fff; }
.icesee-h { font-size: 18px; font-weight: 800; margin: 2px 0 10px; }
.icesee-lbl { width: 120px; font-weight: 650; }
.icesee-k { width: 220px; font-weight: 650; color: rgba(0,0,0,.78); }
.icesee-subtle { color: rgba(0,0,0,.60); font-size: 12px; }
.icesee-status { display:inline-block; padding: 8px 14px; border-radius: 999px; font-weight: 700; border: 1px solid rgba(0,0,0,.10); }
.icesee-idle { background: rgba(0,0,0,.04); }
.icesee-running { background: rgba(16, 122, 255, .12); }
.icesee-done { background: rgba(30, 170, 80, .14); }
.icesee-fail { background: rgba(220, 60, 60, .14); }
</style>
"""
display(W.HTML(css))

# ============================================================
# 11) Layout (your pro UX)
# ============================================================
header = W.HTML(
    "<div class='icesee-wrap'>"
    "<div class='icesee-title'>Run ICESEE examples with one click.</div>"
    "<div class='icesee-subtitle'>Outputs and reports are saved to the wrapper and shown on the right.</div>"
    "</div>"
)

left = W.VBox([
    W.HTML("<div class='icesee-h'>Run settings</div>"),
    W.HBox([W.HTML("<div class='icesee-lbl'>Example:</div>"), example_dd], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Preset:</div>"),  preset_dd],  layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Filter:</div>"),  filter_alg_dd],  layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Output:</div>"),  output_label_dd], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Ens:</div>"),     ens_sl],     layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Seed:</div>"),    seed_in],    layout=W.Layout(gap="12px")),
    W.Box([gen_report], layout=W.Layout(margin="6px 0 0 120px")),
    W.Box([open_latest], layout=W.Layout(margin="0 0 8px 120px")),
    W.HTML("<div class='icesee-subtle' style='margin:8px 0 8px'>Full configuration (from <code>params.yaml</code>)</div>"),
])

params_holder = W.VBox([])
left_card = W.VBox([left, params_holder])
left_card.add_class("icesee-card")

right = W.VBox([
    W.HTML("<div class='icesee-h'>Run log</div>"),
    log_out,
    W.HTML("<div class='icesee-h' style='margin-top:14px'>Results preview</div>"),
    results_out,
])
right_card = W.VBox([right])
right_card.add_class("icesee-card")

actions = W.HBox([run_btn, clear_btn, status_chip], layout=W.Layout(gap="12px"))
actions_card = W.VBox([W.HTML("<div class='icesee-h'>Status</div>"), actions])
actions_card.add_class("icesee-card")

page = W.VBox([
    header,
    W.HBox([left_card, right_card], layout=W.Layout(gap="26px")),
    actions_card
])

display(page)
set_status("idle")
rebuild_for_example()

# Keep template in sync with the quick knobs:
def _sync_knobs(_=None):
    # map filter_alg into enkf-parameters/filter_type etc.
    sync_quick_into_widgets()

filter_alg_dd.observe(_sync_knobs, names="value")
ens_sl.observe(_sync_knobs, names="value")
seed_in.observe(_sync_knobs, names="value")

HTML(value='\n<style>\n.icesee-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial; }\n.ices…